In [1]:
import sys
sys.path.append("./../")
import matplotlib.pyplot as plt
from torchsummary import summary
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score
import math
from PIL import Image
import time
from scipy.io import loadmat as loadmat
from scipy import io
import random
import numpy as np
from thop import profile
import os
import torch 
import torch.utils.data as dataf
import torch.nn as nn
from operator import truediv
import record
import pandas as pd
import seaborn as sns
from mmamba_model import MMamba
from dataset import Multimodal_Dataset_Train, Multimodal_Dataset_Test
import torch.backends.cudnn as cudnn
cudnn.deterministic = True
cudnn.benchmark = True



Bad key axes3d.automargin in file /home/cgw/anaconda3/envs/multi310/lib/python3.10/site-packages/matplotlib/mpl-data/matplotlibrc, line 430 ('axes3d.automargin: False  # automatically add margin when manually setting 3D axis limits')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.8.2/lib/matplotlib/mpl-data/matplotlibrc
or from the matplotlib source distribution

Bad key image.interpolation_stage in file /home/cgw/anaconda3/envs/multi310/lib/python3.10/site-packages/matplotlib/mpl-data/matplotlibrc, line 606 ('image.interpolation_stage: data     # see help(imshow) for options')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.8.2/lib/matplotlib/mpl-data/matplotlibrc
or from the matplotlib source distribution


In [2]:
def get_confusion_matrix(y_test,y_pred, plt_name):
    df_cm = pd.DataFrame(confusion_matrix(y_test, y_pred), range(11),range(11))
    df_cm.columns = ['Trees', 'Grass_Pure', 'Grass_Groundsurface', 'Dirt_And_Sand', 'Road_Materials', 'Water',
                        'Buildings', "Buildings'_Shadow",
                        'Sidewalk', 'Yellow_Curb', 'ClothPanels']
    df_cm = df_cm.rename({0:'Trees', 1:'Grass_Pure', 2:'Grass_Groundsurface', 3:'Dirt_And_Sand', 4:'Road_Materials', 5:'Water',
                        6:'Buildings', 7:"Buildings'_Shadow",
                        8:'Sidewalk', 9:'Yellow_Curb', 10:'ClothPanels'})
    df_cm.index.name = 'Actual'
    df_cm.columns.name = 'Predicted'
    sns.set(font_scale=0.9)#for label size
    sns.heatmap(df_cm, cmap="Blues",annot=True,annot_kws={"size": 16}, fmt='g')
    plt.savefig(''+str(plt_name)+'.eps', format='eps')

def AA_andEachClassAccuracy(confusion_matrix):
    counter = confusion_matrix.shape[0]
    list_diag = np.diag(confusion_matrix)
    list_raw_sum = np.sum(confusion_matrix, axis=1)
    each_acc = np.nan_to_num(truediv(list_diag, list_raw_sum))
    average_acc = np.mean(each_acc)
    return each_acc, average_acc

def reports (xtest,xtest2,ytest,name,model, HSIOnly, iternum):
    pred_y = np.empty((len(ytest)), dtype=np.float32)
    number = len(ytest) // testSizeNumber
    for i in range(number):
        temp = xtest[i * testSizeNumber:(i + 1) * testSizeNumber, :, :]
        temp = temp.cuda()
        temp1 = xtest2[i * testSizeNumber:(i + 1) * testSizeNumber, :, :]
        temp1 = temp1.cuda()
        if HSIOnly:
            temp2 = model(temp)
        else:
            temp2 = model(temp,temp1)
        temp3 = torch.max(temp2, 1)[1].squeeze()
        pred_y[i * testSizeNumber:(i + 1) * testSizeNumber] = temp3.cpu()
        del temp, temp2, temp3,temp1

    if (i + 1) * testSizeNumber < len(ytest):
        temp = xtest[(i + 1) * testSizeNumber:len(ytest), :, :]
        temp = temp.cuda()
        temp1 = xtest2[(i + 1) * testSizeNumber:len(ytest), :, :]
        temp1 = temp1.cuda()
        if HSIOnly:
            temp2 = model(temp)
        else:
            temp2 = model(temp,temp1)
        temp3 = torch.max(temp2, 1)[1].squeeze()
        pred_y[(i + 1) * testSizeNumber:len(ytest)] = temp3.cpu()
        del temp, temp2, temp3,temp1

    pred_y = torch.from_numpy(pred_y).long()
    
    oa = accuracy_score(ytest, pred_y)
    confusion = confusion_matrix(ytest, pred_y)
    each_acc, aa = AA_andEachClassAccuracy(confusion)
    kappa = cohen_kappa_score(ytest, pred_y)
    get_confusion_matrix(ytest, pred_y, 'test_'+str(iternum)+'')
    
    
    
    return confusion, oa*100, each_acc*100, aa*100, kappa*100


In [3]:
os.environ["CUDA_VISIBLE_DEVICES"]="1"
datasetNames = ["Muufl"]
data2Name = 'LIDAR'

num_devices = torch.cuda.device_count()
print(f"Number of available CUDA devices: {num_devices}")

for i in range(num_devices):
    print(f"CUDA Device {i}: {torch.cuda.get_device_name(i)}")
    
device_idx = torch.cuda.current_device()
print(f"Currently active CUDA device: {device_idx}")

batchsize = 64
testSizeNumber = 500
EPOCH = 100
BandSize = 1
LR = 2e-4
FM = 8
HSIOnly = False
FileName = 'MMamba_muufl'
drop_rate = 0.1
depth = 2
token = 11
train_loss = []
KAPPA = []
OA = []
AA = []
ELEMENT_ACC = np.zeros((3, 11))

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

for circle in range(3):
    for datasetName in datasetNames:
        print("---------------------------------- Dataset details for ",datasetName," ---------------------------------------------")
        print('\n')
        try:
            os.makedirs(datasetName)
        except FileExistsError:
            pass
        
        train_dataset = Multimodal_Dataset_Train(Filename=datasetName, MM_Data=data2Name)
        test_dataset = Multimodal_Dataset_Test(Filename=datasetName, MM_Data=data2Name)
        NC = train_dataset.hs_ims.shape[1]
        NCLidar = train_dataset.lid_ims.shape[1]
        Classes = len(torch.unique(train_dataset.lbs))
        length = train_dataset.hs_ims.shape[2] * train_dataset.hs_ims.shape[3]

        train_loader = dataf.DataLoader(train_dataset, batch_size=batchsize, shuffle=True, num_workers=4)
        print("HSI Train data shape = ", train_dataset.hs_ims.shape)
        print(data2Name + " Train data shape = ", train_dataset.lid_ims.shape[1])
        print("Train label shape = ", train_dataset.lbs.shape)

        print("HSI Test data shape = ", test_dataset.hs_ims.shape)
        print(data2Name + " Test data shape = ", test_dataset.lid_ims.shape[1])
        print("Test label shape = ", test_dataset.lbs.shape)

        print("Number of Classes = ", Classes)
        
        patchsize = train_dataset.hs_ims.shape[2]
        TestPatch1 = test_dataset.hs_ims
        TestPatch2 = test_dataset.lid_ims
        TestLabel1 = test_dataset.lbs 
        
        set_seed(42)
        for iterNum in range(1):
            print('\n')
            print("---------------------------------- Model Summary ---------------------------------------------")
            print('\n')

            model = MMamba(FM=FM, NC=NC, NCLidar=NCLidar, Classes=Classes, HSIOnly=HSIOnly, patchsize=patchsize, drop_path=drop_rate, depth=depth, token = token, length = length).cuda()
            summary(model, [(NC, patchsize**2), (NCLidar, patchsize**2)])                    

            # Calculate FLOPs and parameters
            if HSIOnly:
                inputs = torch.randn(1, NC, patchsize**2).cuda()
            else:
                inputs = (torch.randn(1, NC, patchsize**2).cuda(), torch.randn(1, NCLidar, patchsize**2).cuda())
            flops, params = profile(model, inputs=inputs)
            print(f"FLOPs: {flops / 1e6:.3f} MFLOPs")
            

            optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
            loss_func = nn.CrossEntropyLoss()
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
            BestAcc = 0
            Bestepoch = 0

            print('\n')
            print("---------------------------------- Training started for ", datasetName, " ---------------------------------------------")
            print('\n')
            
            torch.cuda.synchronize()
            
            
            # Train the model
            for epoch in range(EPOCH):
                start_train = time.time()
                # if epoch < 50:
                #     weight_decay = 5e-3
                # else:
                #     weight_decay = 1e-5

                # # Reinitialize optimizer with the new weight_decay value
                # optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=weight_decay)

                # # Add scheduler after optimizer is defined
                # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)
                for step, (b_x1, b_x2, b_y) in enumerate(train_loader):

                    # Move train data to GPU
                    b_x1 = b_x1.cuda()
                    b_y = b_y.cuda()
                    if HSIOnly:
                        out1 = model(b_x1)
                        loss = loss_func(out1, b_y)
                    else:
                        b_x2 = b_x2.cuda()
                        out = model(b_x1, b_x2)
                        loss = loss_func(out, b_y)

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    if step % 50 == 0:
                        model.eval()
                        pred_y = np.empty((len(TestLabel1)), dtype='float32')
                        number = len(TestLabel1) // testSizeNumber
                        torch.cuda.synchronize()
                        start_test = time.time()

                        for i in range(number):
                            temp = TestPatch1[i * testSizeNumber:(i + 1) * testSizeNumber, :, :]
                            temp = temp.cuda()
                            
                            temp1 = TestPatch2[i * testSizeNumber:(i + 1) * testSizeNumber, :, :]
                            temp1 = temp1.cuda()
                            if HSIOnly:
                                temp2 = model(temp)
                                temp3 = torch.max(temp2, 1)[1].squeeze()
                                pred_y[i * testSizeNumber:(i + 1) * testSizeNumber] = temp3.cpu()
                                del temp, temp2, temp3
                            else:
                                temp2 = model(temp, temp1)
                                temp3 = torch.max(temp2, 1)[1].squeeze()
                                pred_y[i * testSizeNumber:(i + 1) * testSizeNumber] = temp3.cpu()
                                del temp, temp1, temp2, temp3

                        if (i + 1) * testSizeNumber < len(TestLabel1):
                            temp = TestPatch1[(i + 1) * testSizeNumber:len(TestLabel1), :, :]
                            temp = temp.cuda()
                            temp1 = TestPatch2[(i + 1) * testSizeNumber:len(TestLabel1), :, :]
                            temp1 = temp1.cuda()
                            if HSIOnly:
                                temp2 = model(temp)
                                temp3 = torch.max(temp2, 1)[1].squeeze()
                                pred_y[(i + 1) * testSizeNumber:len(TestLabel1)] = temp3.cpu()
                                del temp, temp2, temp3
                            else:
                                temp2 = model(temp, temp1)
                                temp3 = torch.max(temp2, 1)[1].squeeze()
                                pred_y[(i + 1) * testSizeNumber:len(TestLabel1)] = temp3.cpu()
                                del temp, temp1, temp2, temp3

                        torch.cuda.synchronize()
                        end_test = time.time()

                        pred_y = torch.from_numpy(pred_y).long()
                        accuracy = torch.sum(pred_y == TestLabel1).type(torch.FloatTensor) / TestLabel1.size(0)

                        print('Epoch: ', epoch, '| train loss: %.4f' % loss.data.cpu().numpy(), '| test accuracy: %.4f' % (accuracy*100))
                        train_loss.append(loss.data.cpu().numpy())

                        # Save the best model
                        if accuracy > BestAcc:
                            BestAcc = accuracy
                            Bestepoch = epoch
                            torch.save(model.state_dict(), datasetName + '/net_params2_' + FileName + '.pkl')

                        model.train()
                end_train = time.time()
                        
                scheduler.step()
                print('\nTraining time (in seconds):', (end_train - start_train)-(end_test - start_test))
                print('Testing time (in seconds):', end_test - start_test)
            torch.cuda.synchronize()
           

           
       

            # Load the saved parameters
            model.load_state_dict(torch.load(datasetName + '/net_params2_' + FileName + '.pkl'))

            model.eval()
            confusion, oa, each_acc, aa, kappa = reports(TestPatch1, TestPatch2, TestLabel1, datasetName, model, HSIOnly, circle)
        KAPPA.append(kappa)
        OA.append(oa)
        AA.append(aa)
        ELEMENT_ACC[circle, :] = each_acc
            
        torch.save(model, datasetName + '/best_model_' + FileName + '_BandSize' + str(BandSize) + '_Iter' + str(circle) + '.pt')
        print('\n')
        print("Overall Accuracy = ", oa)
        print("Best Epoch = ", Bestepoch)
        print('\n')
    print("----------" + datasetName + " Training Finished -----------")
    print("\nThe Confusion Matrix on test data")
    record.record_output(OA, AA, KAPPA, ELEMENT_ACC, circle, './' + datasetName + '/' + FileName + '_BandSize' + str(BandSize) + '_Report2_' + datasetName + '.txt')



Number of available CUDA devices: 1
CUDA Device 0: NVIDIA A40
Currently active CUDA device: 0
---------------------------------- Dataset details for  Muufl  ---------------------------------------------


HSI Train data shape =  torch.Size([2684, 64, 11, 11])
LIDAR Train data shape =  2
Train label shape =  torch.Size([2684])
HSI Test data shape =  torch.Size([51003, 64, 11, 11])
LIDAR Test data shape =  2
Test label shape =  torch.Size([51003])
Number of Classes =  11


---------------------------------- Model Summary ---------------------------------------------


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv3d-1       [-1, 18, 56, 11, 11]           1,476
       BatchNorm3d-2       [-1, 18, 56, 11, 11]              36
              ReLU-3       [-1, 18, 56, 11, 11]               0
            Conv2d-4           [-1, 64, 11, 11]         145,216
            Conv2d-5           [-1, 64, 11

In [4]:
train_loss = np.asarray(train_loss)
np.save('HSILiDAR_channel_train_loss.npy', train_loss)